In [8]:
# Jupyter Notebook cell (Python 3)
# -----------------------------------------------------------
# This code demonstrates how to process an input video to detect
# and draw hand meshes (skeletons) with MediaPipe's Hand Landmarker,
# and then save the resulting annotated frames to a new video file
# with the same extension but a suffix of "_handskel".
#
# Requirements:
#   - mediapipe (>= 0.9.1 or the latest version that supports 'tasks')
#   - opencv-python
#   - numpy
#   - (optional) plotly, if you want to visualize images interactively
#
# Usage:
#   1. Place your input video in the same folder or provide its full path.
#   2. Update 'input_video_path' below with your video file name or path.
#   3. Run this cell. The output video will be saved in the same directory
#      with the suffix "_handskel" added to the name.

import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import os

def process_video_with_hand_landmarker(input_video_path: str, 
                                       model_path: str = "/home/robotics/Downloads/hand_landmarker.task",
                                       max_num_hands: int = 2):
    """
    Processes the given video file to detect hand landmarks using MediaPipe's
    HandLandmarker (Tasks API), draws hand skeletons on each frame, and saves
    the result to an output video file with '_handskel' appended to the name.

    Args:
        input_video_path (str): Path to the input video file.
        model_path (str): Path to the MediaPipe Hand Landmarker model.
                          Default is 'hand_landmarker.task'.
        max_num_hands (int): Max number of hands to detect per frame.
    """
    # ----------------------------------------------------------------
    # 1. Prepare the output video path
    #    e.g., if the input is "example.mp4", the output will be "example_handskel.mp4"
    # ----------------------------------------------------------------
    base_name, ext = os.path.splitext(input_video_path)
    output_video_path = f"{base_name}_handskel{ext}"

    # ----------------------------------------------------------------
    # 2. Set up the video capture and writer
    # ----------------------------------------------------------------
    cap = cv2.VideoCapture(input_video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video file {input_video_path}")
        return
    
    # Get the video properties
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v') if ext.lower() == '.mp4' else cv2.VideoWriter_fourcc(*'XVID')
    
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
    
    # ----------------------------------------------------------------
    # 3. Initialize the MediaPipe Hand Landmarker
    # ----------------------------------------------------------------
    BaseOptions = mp.tasks.BaseOptions
    HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
    RunningMode = mp.tasks.vision.RunningMode
    
    options = HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=RunningMode.VIDEO,
        num_hands=max_num_hands
    )
    
    with mp.tasks.vision.HandLandmarker.create_from_options(options) as hand_landmarker:
        frame_index = 0
        
        while True:
            success, frame_bgr = cap.read()
            if not success:
                break  # End of video

            # Convert the BGR frame to RGB for MediaPipe
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

            # Create a MediaPipe Image object
            mp_image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=frame_rgb
            )

            # The 'timestamp_ms' should be set to the current frame timestamp in milliseconds
            frame_timestamp_ms = int((1000.0 / fps) * frame_index)

            # Detect hands in this frame
            detection_result = hand_landmarker.detect_for_video(
                mp_image, frame_timestamp_ms
            )
            
            # ----------------------------------------------------------------
            # 4. Draw landmarks on the frame
            # ----------------------------------------------------------------
            if detection_result.hand_landmarks:
                # Go through each detected hand
                for idx, landmarks in enumerate(detection_result.hand_landmarks):
                    # Each 'landmarks' is a list of 21 normalized landmark points in [0,1] range
                    # We must scale them to the image dimensions before drawing
                    for landmark in landmarks:
                        # Convert normalized landmark [x,y] into image coordinates
                        x_px = int(landmark.x * width)
                        y_px = int(landmark.y * height)

                        # Draw a small circle at the landmark
                        cv2.circle(frame_bgr, (x_px, y_px), 5, (0, 255, 0), -1, cv2.LINE_AA)
                    
                    # Draw connections (hand skeleton)
                    # The typical MediaPipe hand connections for 21 landmarks:
                    # HAND_CONNECTIONS = [
                    #     (0,1), (1,2), (2,3), (3,4),    # Thumb
                    #     (0,5), (5,6), (6,7), (7,8),    # Index finger
                    #     (5,9), (9,10), (10,11), (11,12),  # Middle finger
                    #     (9,13), (13,14), (14,15), (15,16), # Ring finger
                    #     (13,17), (17,18), (18,19), (19,20) # Pinky
                    # ]
                    HAND_CONNECTIONS = [
                        (0,1), (1,2), (2,3), (3,4),    # Thumb
                        (0,5), (5,6), (6,7), (7,8),    # Index finger
                        (0,9), (5,9), (9,10), (10,11), (11,12),  # Middle finger
                        (0,13), (9,13), (13,14), (14,15), (15,16), # Ring finger
                        (0,17), (13,17), (17,18), (18,19), (19,20) # Pinky
                    ]
                    # Draw lines between connected landmarks
                    for connection in HAND_CONNECTIONS:
                        start_idx, end_idx = connection
                        x_start = int(landmarks[start_idx].x * width)
                        y_start = int(landmarks[start_idx].y * height)
                        x_end   = int(landmarks[end_idx].x * width)
                        y_end   = int(landmarks[end_idx].y * height)
                        cv2.line(frame_bgr, (x_start, y_start), (x_end, y_end),
                                 color=(0, 255, 255), thickness=2, lineType=cv2.LINE_AA)

            # ----------------------------------------------------------------
            # 5. Write the annotated frame to the output
            # ----------------------------------------------------------------
            out.write(frame_bgr)
            frame_index += 1
    
    # Clean up
    cap.release()
    out.release()
    print(f"Output video saved to: {output_video_path}")

# ----------------------------------------------------------------
# Example usage:
# ----------------------------------------------------------------
# Update 'input_video_path' to your actual video file name or path
input_video_path = "/home/robotics/Desktop/Pilot_Data/Dummy Participant 5/camera1.avi"  # e.g. "sample_video.mp4"
process_video_with_hand_landmarker(input_video_path)


I0000 00:00:1740484396.152095   48909 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1740484396.241656   50934 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 560.35.05), renderer: NVIDIA GeForce RTX 4060 Ti/PCIe/SSE2
W0000 00:00:1740484396.258317   50946 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1740484396.281724   50937 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Output video saved to: /home/robotics/Desktop/Pilot_Data/Dummy Participant 5/camera1_handskel.avi
